# Create helper methods to:
 1. Load the CoDocGen model to generate documentation for a function
 2. Method that uses the CoDocModel model to generate documentation of the given code (as string)
 3. Load the the-stack dataset and extract only the C++ and python code from it.
 4. load the github/tree-sitter to create abstract syntx trees of given code and lanugage
 5. Create ASTs for the given code.

 Install all the necessary packages

In [1]:
# Required for code doc generation
!pip install transformers torch accelerate  datasets

In [2]:
# required for AST generator and comparator
!pip install tree-sitter tree-sitter-languages tree-sitter-cpp tree-sitter-python tree-sitter-cpp numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 13.5 MB/s eta 0:00:00


In [3]:
# Upgrade torchao to a compatible version
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


# CodeBERTScorer Package Installation

In [4]:
!pip install sentence-transformers

In [6]:
from google.colab import userdata
import os

# Try to retrieve the Hugging Face API key from Colab's secrets manager
HUGGING_FACE_KEY = userdata.get('HUGGING_FACE_KEY')

if HUGGING_FACE_KEY is None:
    print("WARNING: Hugging Face API key (HUGGING_FACE_KEY) not found in Colab secrets.")

    # Prompt user for input if not found in secrets
    HUGGING_FACE_KEY = input("Please enter your Hugging Face API Key: ")
    if not HUGGING_FACE_KEY:
        print("Hugging Face API Key was not provided. Some models might not load correctly.")
        import sys
        sys.exit()
    else:
        os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
        print("Hugging Face API key received from input.")
else:
    # Set the environment variable for Hugging Face if found in secrets
    os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
    print("Hugging Face API key loaded successfully from secrets.")

Hugging Face API key loaded successfully from secrets.


# All models will be singleton so create base metaclass for the singleton

In [7]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):

        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(
                *args,
                **kwargs
            )

        return cls._instances[cls]

 # Singleton for the CodeGeneration

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch

class CodeDocumentationGenerator(metaclass=SingletonMeta):
    def __init__(self):
        """
        Constructor.

        IMPORTANT:
        Since SingletonMeta returns the same object every time,
        __init__ may be called multiple times.

        Therefore we guard against reinitialization.
        """

        if hasattr(self, "_initialized"):
            return

        self._initialized = True
        #self._model_name = "CoDoCGen/CoDoCGen-7B"
        self._model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"
        self._load_codocgen_model()


    def _load_codocgen_model(self):
      """Loads the CoDoCGen model and tokenizer."""
      self._model = AutoModelForCausalLM.from_pretrained(self._model_name,
                                                  dtype=torch.float16,
                                                  device_map="auto")

      self._tokenizer = AutoTokenizer.from_pretrained(self._model_name)


    def generate_documentation(self, code: str, max_length=512, num_return_sequences=1, prompt=None) -> list:
      """
        Generates documentation for a given code snippet using the CoDoCGen model.

      Args:
          code (str): The code for which to generate documentation.
          max_length (int): The maximum length of the generated documentation.
          num_return_sequences (int): The number of different documentation sequences to generate.

      Returns:
          list: A list of generated documentation strings.
      """
      if prompt is None:
        prompt = f"generate good detailed documentation for what this software code does, do not include a pseudo code or example usage, just the intent of what the program should do, if it follows a design pattern, what actions to take under what conditions. The first line of the documentation should start with 'A software program in ProgLang, where ProgLan is the programming language of the program: {code}"
      else:
        prompt = f"{prompt}"

      messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
      ]
      text = self._tokenizer.apply_chat_template( messages, tokenize=False, add_generation_prompt=True)
      model_inputs = self._tokenizer([text], return_tensors="pt").to(self._model.device)

      generated_ids = self._model.generate(**model_inputs, max_new_tokens=512)
      generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids) ]

      response = self._tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
      return response

In [9]:
if 'flag_demo_code_generation' in globals() and flag_demo_code_generation:
  # Test the documentation generation
  sample_code = """
  def factorial(n):
      if n == 0:
          return 1
      else:
          return n * factorial(n-1)
  """

  print("Generating documentation for the following code:")
  print(sample_code)

### Test the Code Documentation Generator

In [10]:
if 'flag_demo_code_generation' in globals() and flag_demo_code_generation:
  generator = CodeDocumentationGenerator()

In [11]:
if 'flag_demo_code_generation' in globals() and flag_demo_code_generation:
  print("\nGenerated Documentation 2:")
  documentation = generator.generate_documentation(sample_code)
  print(documentation)

# Obsolete TestLoad and Filter Code Dataset

A function to load a `bigcode/the-stack` dataset and filter it to include only C++ and Python code. The `language` column has the type of laguage

In [12]:
if 'flag_dataset_load_and_filtering' in globals() and flag_dataset_load_and_filtering:
  from datasets import load_dataset, interleave_datasets

  def load_and_filter_code_dataset(languages:list =None):
      """
      Loads a code dataset and filters it by specified languages.

      Args:
          dataset_name (str): The name of the dataset to load (e.g., "codeparrot/github-code").
          languages (list): A list of programming languages to filter by (e.g., ['C++', 'Python']).
                            If None, no language filtering is applied.

      Returns:
          datasets.Dataset: The filtered dataset.
      """
      dataset_name ="bigcode/the-stack-dedup"
      languages_and_dataset = [
                                {
                                    'language':'python',
                                    'dataset':None
                                },
                                {
                                    'language':'cpp',
                                    'dataset':None
                                }
                              ]

      print(f"Loading dataset: {dataset_name}")
      train_dataset = None

      for i in range(len(languages_and_dataset)):
        language = languages_and_dataset[i]['language']
        print(f"Fetching dataset for '{language}' language")
        # 2. Merge them into one combined stream
        # probabilities=[0.5, 0.5] mixes them evenly (1 Python, 1 C++, 1 Python...)
        languages_and_dataset[i]['dataset'] = load_dataset(dataset_name,
                                        data_dir = f"data/{language}",
                                        split="train",
                                        streaming=True,
                                          token=True)


      return languages_and_dataset

### Obsolete Test: Loading and Filtering Code

Use the `load_and_filter_code_dataset` function to get only C++ and Python code from the `codeparrot/github-code` dataset.

In [13]:
if 'flag_dataset_load_and_filtering' in globals() and flag_dataset_load_and_filtering:
  try:
      cpp_python_dataset = load_and_filter_code_dataset()

      print(next(iter(cpp_python_dataset[0]['dataset'])))
      print(next(iter(cpp_python_dataset[1]['dataset'])))
      # print("\nFirst example from filtered dataset (showing language and a snippet of code):")
      # first_example_filtered = next(iter(cpp_python_dataset))
      # print(f"---\nLanguage: {first_example_filtered.get('lang', 'N/A')}\nCode Snippet: {first_example_filtered.get('content', 'N/A')[:200]}...")

      # The original intent was to get 5 examples, but for streaming datasets,
      # directly indexing or taking len() can be problematic. Iterating explicitly is safer.
      # Let's just confirm the first example for now.

  except RuntimeError as e:
      if "Dataset scripts are no longer supported" in str(e):
          print(f"\nError loading dataset: {e}")
          print("\nIt appears the `codeparrot/github-code` dataset cannot be loaded directly via script anymore.")
          print("To fix this, please modify the `load_and_filter_code_dataset` function in cell `CnarbxKBNomR`.")
          print("You might need to specify a `config_name` (e.g., 'all' or 'code_x_m') and potentially use `streaming=True` if the dataset is very large, like so:")
          print("    `dataset = load_dataset(dataset_name, 'all', split=\"train\", streaming=True)`")
          print("Alternatively, you might need to find a different version of the dataset or a more compatible dataset for demonstration purposes.")
      else:
        raise e

In [15]:
if 'flag_dataset_load_and_filtering' in globals() and flag_dataset_load_and_filtering:
  print("\nFirst example from cpp_python_dataset (using next(iter())):\n")
  first_example = next(iter(cpp_python_dataset))
  print(first_example)

# Singleton for AST Generation and Comparison

The `AST` class leverages `tree-sitter` to generate Abstract Syntax Trees (ASTs) from code snippets and provides a method to compare two ASTs. It is implemented as a singleton to ensure a single instance manages the language parsers and potentially future comparison models.

For generating the AST the input code can be a string or bytes. If string then convert to bytes before processing with tree sitter.


In [16]:
import io
from contextlib import redirect_stdout
from typing import Union
from tree_sitter import Tree
import numpy as np

class AST(metaclass=SingletonMeta):
    def __init__(self):
        """
        Constructor for the AST singleton class.
        Initializes AST generator and comparator.
        Guards against reinitialization.
        """
        if hasattr(self, "_initialized"):
            return

        self._initialized = True
        self._load_ast_generator()

    def _load_ast_generator(self):
        """
        Loading AST generation models/parsers.
        """
        from tree_sitter import Language, Parser
        import tree_sitter_cpp as tscpp
        import tree_sitter_python as tspy

        # Load the language dynamically using tree-sitter-languages.
        # This function directly returns a tree_sitter.Language object.
        self._ast_cpp_parser = Parser()
        self._ast_cpp_parser.language = Language(tscpp.language())
        self._ast_python_parser = Parser()
        self._ast_python_parser.language = Language(tspy.language())


    def generate_ast(self, code: Union[str, bytes], language_name: str) -> Tree:
        """
          Generates an Abstract Syntax Tree (AST) for a given code snippet
          using tree-sitter for the specified language.

          Args:
          code (Union[str, bytes]): The code for which to generate the AST. Can be a string or bytestring.
          language_name (str): The name of the programming language (e.g., 'python', 'cpp').

        Returns:
        A tuple where:
        first element: tree_sitter.Tree: The generated AST.
        second element: string reresentation of the AST
        """

        # Check if the language is supported
        if language_name.lower() not in ['cpp', 'c++', 'python']:
          raise ValueError(f"Supported programming languages are 'C++', 'Python'. '{language_name}' is not supported!")

        # Encode string to bytes if necessary
        if isinstance(code, str):
          code = code.encode('utf-8')

        # Load the language dynamically using tree-sitter-languages.
        # This function directly returns a tree_sitter.Language object.
        ast_tree = None
        if language_name.lower() in ['cpp', 'c++']:
          ast_tree = self._ast_cpp_parser.parse(code)

        elif language_name.lower() == 'python':
          ast_tree = self._ast_python_parser.parse(code)

        else:
          raise ValueError(f"Runtime environment error for language {language_name}")

        return ast_tree

    def compare_ast(self, ast1: Tree, ast2: Tree) -> float:
        """
        Do not compare the raw Tree-Sitter trees, because
        'def add(a,b):' and 'def add( a , b):' althought same have different ASTs.
        instead normalize the representation.
        So (a quick search gave this):
        Step 1:  Keep node types, structure and remove puncutation and raw tokens.
        Step 2: Convet tree to ordered node list or S-expression
        Step 3: Compare using Jaccard similarity on AST node sequences.
                Jaccard Simnilrity = (Intersection / Union)
        Compares two ASTs (ASTs using structural node similarity) and returns a similarity score.
        Args:
            ast1 (Tree): The first AST tree_sitter.Tree object.
            ast2 (Tree): The second AST tree_sitter.Tree object.

        Returns:
            float: A similarity score between 0.0 and 1.0, where 1.0 means identical.
        """
        def extract_ast_nodes(node):
          """
            Extracts only structural node types from Tree-sitter AST.
            This removes syntax noise like brackets, commas, etc.
          """
          nodes = []

          def dfs(n):
            # keep only meaningful AST node types
            if n.child_count > 0:
                nodes.append(n.type)

            for child in n.children:
                dfs(child)

          dfs(node)
          return nodes

        nodes1 = extract_ast_nodes(ast1.root_node)
        nodes2 = extract_ast_nodes(ast2.root_node)

        set1 = set(nodes1)
        set2 = set(nodes2)

        intersection = len(set1 & set2)
        union = len(set1 | set2)

        if union == 0:
          return 0.0

        return intersection / union

    def to_str(self, tree: Tree)->str:
      '''
      Generate string format for the tree
      '''
      f = io.StringIO()
      with redirect_stdout(f):
          self.print_ast_tree(tree)
      return f.getvalue()

    def print_ast_tree(self, tree: Tree, indent=0):
      """
      Helper function to print AST nodes recursively with indentation.
      """
      def _print_node_recursive(node, current_indent):
          print(f"{_get_indent_string(current_indent)}{node.type} [start={node.start_point}, end={node.end_point}]")
          for child in node.children:
              _print_node_recursive(child, current_indent + 1)

      def _get_indent_string(current_indent):
          return '  ' * current_indent

      _print_node_recursive(tree.root_node, indent)


### Test: Generating ASTs

Test the AST generator and comparator for C++ and python using the AST class.

In [17]:
if 'flag_ast_generation_and_test' in globals() and flag_ast_generation_and_test:
  # Example 1: Python Code
  python_code = """
  def greet(name):
      print(f"Hello, {name}!")
  """


  ast = AST()
  print("Generating AST for Python code:")
  python_ast = ast.generate_ast(python_code, 'python')
  ast_str = ast.to_str(python_ast)
  print("--\n", ast_str)
  if python_ast:
      ast.print_ast_tree(python_ast)

  print("\n" + "="*50 + "\n")

  # Example 2: C++ Code
  cpp_code = b"""
  #include <iostream>

  int main() {
      std::cout << "Hello from C++!" << std::endl;
      return 0;
  }
  """

  print("Generating AST for C++ code:")
  cpp_ast = ast.generate_ast(cpp_code, 'cpp')
  if cpp_ast:
      ast.print_ast_tree(cpp_ast)

  score = ast.compare_ast(python_ast, cpp_ast)
  print(f"\nAST Similarity Score: {score}")

# GraphCodeBERTScorer for Semantic Code Similarity

The `GraphCodeBERTScorer` class is a singleton that uses a pre-trained `GraphCodeBERT` model (via `sentence-transformers`) to generate semantic embeddings for code snippets. It then calculates the cosine similarity between these embeddings to provide a semantic similarity score between two pieces of code. This is useful for tasks where AST comparison might miss semantic equivalence due to structural differences.

In [35]:
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
import numpy as np
import torch # Import torch for float16

class GraphCodeBERTScorer(metaclass=SingletonMeta):
    def __init__(self):
        """
        Constructor for the GraphCodeBERTScorer singleton class.
        Initializes the GraphCodeBERT model for semantic code similarity scoring.
        Guards against reinitialization.
        """
        if hasattr(self, "_initialized"):
            return

        self._initialized = True
        print("Initializing GraphCodeBERTScorer...")
        self._load_model()
        print("GraphCodeBERTScorer initialized successfully.")

    def _load_model(self):
        """
        Loads the pre-trained GraphCodeBERT model for generating code embeddings.
        """
        # Using GraphCodeBERT-base from Hugging Face via sentence-transformers
        self.model = SentenceTransformer('microsoft/graphcodebert-base')
        # Move model to GPU and convert to float16 for reduced memory usage
        self.model.to('cuda', dtype=torch.float16)

    def score(self, code_1: str, code_2: str) -> float:
        """
        Calculates the semantic similarity score between two code snippets
        using GraphCodeBERT embeddings and cosine similarity.

        Args:
            code_1 (str): The first code snippet.
            code_2 (str): The second code snippet.

        Returns:
            float: A similarity score between 0.0 and 1.0, where 1.0 means identical semantics.
        """
        if not isinstance(code_1, str) or not isinstance(code_2, str):
            raise ValueError("Both code snippets must be strings.")

        # Generate GraphCodeBERT embeddings for both code snippets
        # Ensure encoding happens on the correct device
        embeddings = self.model.encode([code_1, code_2], convert_to_tensor=True, device=self.model.device)

        # Calculate cosine similarity between the embeddings
        # Cosine distance is 1 - cosine similarity, so similarity = 1 - distance
        similarity_score = 1 - cosine(embeddings[0].cpu().numpy(), embeddings[1].cpu().numpy())

        return float(similarity_score)


# Singleton for Filtered and Enriched Code Dataset

This `FilteredDataset` class acts as a singleton to efficiently manage access to and processing of the `bigcode/the-stack-dedup` dataset. It specifically filters for C++ and Python code, and for each valid code snippet, it generates detailed documentation using the `CodeDocumentationGenerator` and constructs an Abstract Syntax Tree (AST) using the `AST` processor. This enrichment happens dynamically as you iterate through the dataset, providing a stream of ready-to-use data for tasks like code generation or analysis.

In [41]:
from datasets import load_dataset
from typing import Iterator, Dict, Any, Union
from tree_sitter import Tree # Import Tree for type hinting in AST class, though not directly used in FilteredDataset

class FilteredDataset(metaclass=SingletonMeta):
    _initialized = False # Class-level flag for singleton initialization

    def __init__(self):
        """
        Constructor for the FilteredDataset singleton class.
        Initializes dataset loading and processing components.
        """
        if FilteredDataset._initialized:
            print("FilteredDataset already initialized. Returning existing instance.")
            return
        FilteredDataset._initialized = True

        print("Initializing FilteredDataset singleton...")
        self._local_documentation_cache: Dict[str, str] = {} # Cache for local documentation updates
        self._local_scores_cache: Dict[str, Dict[str, float]] = {} # Cache for storing scores
        self._python_dataset_stream = None
        self._cpp_dataset_stream = None
        self._python_iter = None # Internal iterator for the combined stream
        self._cpp_iter = None # Internal iterator for the combined stream
        self._current_dataset_selector = 0 # 0 for Python, 1 for C++
        self._load_datasets() # Initial loading of the datasets

        print("FilteredDataset initialized successfully.")

    def _convert_max_stars_count(self, example: Dict[str, Any]) -> Dict[str, Any]:
        """
        Helper function to convert 'max_stars_count' to float64, handling missing or invalid values.
        """
        if 'max_stars_count' in example and example['max_stars_count'] is not None:
            try:
                example['max_stars_count'] = float(example['max_stars_count'])
            except (ValueError, TypeError):
                example['max_stars_count'] = None # Assign None for conversion errors
        return example

    def _load_datasets(self):
        """
        Helper method to load the datasets separately and initialize their iterators.
        This also applies the 'max_stars_count' conversion.
        """
        print("Loading Python dataset...")
        python_raw_stream = load_dataset("bigcode/the-stack-dedup", data_dir="data/python", split="train", streaming=True, token=True)
        self._python_dataset_stream = python_raw_stream.map(self._convert_max_stars_count)

        print("Loading C++ dataset...")
        cpp_raw_stream = load_dataset("bigcode/the-stack-dedup", data_dir="data/cpp", split="train", streaming=True, token=True)
        self._cpp_dataset_stream = cpp_raw_stream.map(self._convert_max_stars_count)

        # Initialize iterators for the main __iter__ method
        self._python_iter = iter(self._python_dataset_stream)
        self._cpp_iter = iter(self._cpp_dataset_stream)
        self._current_dataset_selector = 0 # Reset selector for alternating iteration

    def reset_iterator(self):
        """
        Resets the dataset iterator to the beginning, allowing re-iteration from the start.
        For streaming datasets, this involves re-initializing the underlying dataset streams.
        """
        print("Resetting dataset streams and iterators...")
        self._load_datasets() # Re-call to reload the streams and create new iterators

    def _enrich_record(self, example: Dict[str, Any]) -> Dict[str, Any]:
        """
        Helper method to apply common enrichment logic to a single record.
        This includes adding language, documentation (from cache), score placeholders,
        and a flag indicating if the code is processable.
        """
        code_content = example.get('content')
        original_language = example.get('lang')
        hexsha = example.get('hexsha')

        example['language'] = original_language

        # Prioritize local documentation cache
        example['documentation'] = self._local_documentation_cache.get(hexsha, "")

        # Initialize score fields
        example['ast_score'] = self._local_scores_cache.get(hexsha, {}).get('ast_score', None)
        example['graphcodebert_score'] = self._local_scores_cache.get(hexsha, {}).get('graphcodebert_score', None)
        example['average_score'] = self._local_scores_cache.get(hexsha, {}).get('average_score', None)

        example['ast_tree'] = None # AST object itself is not serializable
        example['ast_string_representation'] = ""

        # Determine if the code is processable
        if code_content and original_language and isinstance(code_content, str): # and 50 < len(code_content) < 2000:
            example['is_processable_code'] = True
        else:
            example['is_processable_code'] = False
            if hexsha not in self._local_documentation_cache:
                example['documentation'] = "Skipped: Code content or language invalid/missing."
            example['ast_string_representation'] = "Skipped: Code content or language invalid/missing."
        return example

    def __iter__(self) -> Iterator[Dict[str, Any]]:
        """
        Iterates over the combined (Python and C++) streaming dataset by alternating.
        Enriches each record with language, documentation from local cache (if any),
        score placeholders, and a flag indicating if the code is processable.

        Yields:
            Dict[str, Any]: An enriched dictionary representing a single record from the dataset.
        """
        print("Starting iteration over combined dataset (alternating Python and C++) via __iter__...")
        # The iterators _python_iter and _cpp_iter are managed by _load_datasets/reset_iterator
        # and maintain their state across calls to next() for a single __iter__ session.
        while self._python_iter is not None or self._cpp_iter is not None:
            example = None
            if self._current_dataset_selector == 0: # Try to get from Python dataset
                if self._python_iter:
                    try:
                        example = next(self._python_iter)
                        self._current_dataset_selector = 1 # Switch to C++ for next iteration
                    except StopIteration:
                        self._python_iter = None # Python stream exhausted
                        self._current_dataset_selector = 1 # Try C++ next if Python is exhausted
                else: # Python already exhausted, switch to C++
                    self._current_dataset_selector = 1

            if example is None and self._current_dataset_selector == 1: # Try to get from C++ dataset (either after Python or directly)
                if self._cpp_iter:
                    try:
                        example = next(self._cpp_iter)
                        self._current_dataset_selector = 0 # Switch to Python for next iteration
                    except StopIteration:
                        self._cpp_iter = None # C++ stream exhausted
                        self._current_dataset_selector = 0 # Try Python next if C++ is exhausted
                else: # C++ already exhausted, switch to Python
                    self._current_dataset_selector = 0

            if example is None: # Both streams might be exhausted or only one was active and now exhausted
                if self._python_iter is None and self._cpp_iter is None:
                    break # Both exhausted, stop iteration
                else:
                    # One stream exhausted, but the other might still have data.
                    # The selector should naturally point to the non-exhausted one.
                    # We just continue the loop to try fetching from the other stream.
                    continue

            yield self._enrich_record(example)

    def get_python_stream_iterator(self) -> Iterator[Dict[str, Any]]:
        """
        Returns a fresh iterator for the Python dataset stream, applying enrichment to each record.
        This iterator is independent of the main alternating iterator.
        """
        if self._python_dataset_stream is None:
            # If streams are not loaded, load them
            self._load_datasets()
        # Return a new iterator each time this method is called to allow for independent iteration
        return (self._enrich_record(record) for record in iter(self._python_dataset_stream))

    def get_cpp_stream_iterator(self) -> Iterator[Dict[str, Any]]:
        """
        Returns a fresh iterator for the C++ dataset stream, applying enrichment to each record.
        This iterator is independent of the main alternating iterator.
        """
        if self._cpp_dataset_stream is None:
            # If streams are not loaded, load them
            self._load_datasets()
        # Return a new iterator each time this method is called to allow for independent iteration
        return (self._enrich_record(record) for record in iter(self._cpp_dataset_stream))

    def update_documentation(self, record_identifier: str, new_documentation: str):
        """
        Updates the documentation for a specific record locally within the dataset instance.
        This updated documentation will be returned by subsequent iterations when that
        record's `record_identifier` (hexsha) is encountered.

        Args:
            record_identifier (str): The unique identifier (e.g., 'hexsha') of the record to update.
            new_documentation (str): The new documentation string to associate with the record.
        """
        if not isinstance(record_identifier, str):
            raise TypeError("record_identifier must be a string (e.g., 'hexsha').")
        self._local_documentation_cache[record_identifier] = new_documentation
        print(f"Documentation for record '{record_identifier}' updated locally.")

    def update_scores(self, record_identifier: str, ast_score: float, graphcodebert_score: float, average_score: float):
        """
        Updates the scores for a specific record locally within the dataset instance.

        Args:
            record_identifier (str): The unique identifier (e.g., 'hexsha') of the record to update.
            ast_score (float): The AST similarity score.
            graphcodebert_score (float): The GraphCodeBERT semantic similarity score.
            average_score (float): The average of AST and GraphCodeBERT scores.
        """
        if not isinstance(record_identifier, str):
            raise TypeError("record_identifier must be a string (e.g., 'hexsha').")
        self._local_scores_cache[record_identifier] = {
            'ast_score': ast_score,
            'graphcodebert_score': graphcodebert_score,
            'average_score': average_score
        }
        print(f"Scores for record '{record_identifier}' updated locally: AST={ast_score:.4f}, GCB={graphcodebert_score:.4f}, Avg={average_score:.4f}")

# Efficient Filtering and Caching of Streaming Data

For large streaming datasets, repeatedly iterating and filtering can still be slow. Use `datasets` library features for more efficient filtering and to cache a *subset* of your data if you need to iterate over the same items multiple times:

1.  **Direct Filtering with `IterableDataset.filter()`**: You can apply a `filter()` method directly to `IterableDataset` objects. This is efficient as it processes data on-the-fly without loading the entire stream into memory.

2.  **Materializing a Filtered Subset**: If you need to repeatedly access a *specific, small subset* of the filtered data, it's beneficial to materialize it into a non-streaming `Dataset` and save it to disk. This avoids re-fetching and re-processing from the streaming source every time you iterate or reset.

## Materializing and Enriching the Full Dataset

To persist the entire dataset with generated documentation and ASTs, and to incorporate any local documentation updates, we'll use a function that iterates through the streaming `FilteredDataset`, performs the enrichment, and saves the result as a non-streaming `datasets.Dataset` to disk. This approach is memory-efficient for large datasets by leveraging `datasets.Dataset.from_generator`.

In [24]:
import os
from datasets import Dataset, IterableDataset
from tqdm import tqdm
from functools import partial
from typing import Dict, Any, Optional

def _enrich_single_record(
    example: Dict[str, Any],
    local_doc_cache: Optional[Dict[str, str]] = None # Pass the cache for overrides
) -> Dict[str, Any]:
    """
    Helper to prepare a single record for materialization. It applies local documentation overrides if any.
    Documentation and AST generation are assumed to be done by the user before materialization
    or are handled by the FilteredDataset's __iter__ for initial defaults.
    """
    hexsha = example.get('hexsha')

    # Prioritize local cache for documentation if available
    if local_doc_cache and hexsha in local_doc_cache:
        example['documentation'] = local_doc_cache[hexsha]
    # else, example['documentation'] already contains the default from FilteredDataset.__iter__

    # ast_tree is non-serializable and should always be removed before saving
    example.pop('ast_tree', None)

    # ast_string_representation should already be set by FilteredDataset.__iter__
    # (either empty or 'Skipped...') and is not generated here, as requested.

    return example


def materialize_and_enrich_dataset(
    filtered_dataset_instance: FilteredDataset,
    output_cache_path: str = "./enriched_full_dataset",
    force_reprocess: bool = False,
    num_samples_to_process: Optional[int] = None # Limit for testing or smaller datasets
) -> Dataset:
    """
    Materializes the entire streaming dataset (or a subset), ensuring local documentation overrides
    are applied, and caches the dataset to disk using Dataset.from_generator for memory efficiency.
    Documentation and AST generation are externalized to the user.

    Args:
        filtered_dataset_instance (FilteredDataset): An instance of the FilteredDataset singleton.
        output_cache_path (str): Directory where the enriched dataset will be saved/loaded.
        force_reprocess (bool): If True, re-process and overwrite existing cache.
        num_samples_to_process (int, optional): If provided, processes only this many samples.
                                                Useful for testing with large datasets.

    Returns:
        datasets.Dataset: The fully materialized and enriched dataset.
    """
    os.makedirs(output_cache_path, exist_ok=True)

    if os.path.exists(os.path.join(output_cache_path, 'state.json')) and not force_reprocess:
        print(f"Loading existing enriched dataset from {output_cache_path}...")
        return Dataset.load_from_disk(output_cache_path)

    print(f"Materializing and enriching dataset to {output_cache_path}...")

    local_doc_cache = filtered_dataset_instance._local_documentation_cache # Get the current local overrides

    enrich_func = partial(
        _enrich_single_record,
        local_doc_cache=local_doc_cache
    )

    def generator_function():
        filtered_dataset_instance.reset_iterator() # Start from clean stream
        processed_count = 0
        # Use tqdm to show progress for large operations
        for example in tqdm(filtered_dataset_instance, desc="Materializing records"): # Assuming filtered_dataset_instance itself is iterable
            if num_samples_to_process is not None and processed_count >= num_samples_to_process:
                break

            enriched_example = enrich_func(example)
            yield enriched_example
            processed_count += 1

    print("Creating enriched dataset from generator (this may take a long time for full dataset)...")
    materialized_dataset = Dataset.from_generator(generator_function)

    print(f"Saving enriched dataset to {output_cache_path}...")
    materialized_dataset.save_to_disk(output_cache_path)
    print("Enriched dataset saved successfully.")

    return materialized_dataset


In [25]:
if False:
  # Example usage:
  # Ensure FilteredDataset is initialized
  filtered_dataset = FilteredDataset()

  # --- Demonstrate local documentation update before full materialization ---
  print("\n--- Demonstrating local documentation update ---")
  # Fetch a sample record to get its hexsha
  sample_record = None
  filtered_dataset.reset_iterator()
  for i, record in enumerate(filtered_dataset):
      if record['is_processable_code'] and record.get('hexsha'):
          sample_record = record
          break

  if sample_record:
      original_hexsha = sample_record['hexsha']
      print(f"Original documentation for record {original_hexsha[:10]}...: {sample_record['documentation']}")
      new_doc = "This is a manually updated documentation for a test record to check local cache handling."
      filtered_dataset.update_documentation(original_hexsha, new_doc)
      print(f"Updated documentation locally for record {original_hexsha[:10]}...")

      # Iterate again to confirm the local update is visible
      print("Confirming local update in next iteration:")
      filtered_dataset.reset_iterator()
      for i, record in enumerate(filtered_dataset):
          if record.get('hexsha') == original_hexsha:
              print(f"Fetched record {original_hexsha[:10]}... has documentation: {record['documentation']}")
              break

  # --- Now, materialize and enrich the dataset ---
  # For demonstration, let's process a small number of samples
  # For the full dataset, remove num_samples_to_process=100
  num_samples_to_process_full = 100
  full_enriched_dataset = materialize_and_enrich_dataset(
      filtered_dataset,
      num_samples_to_process=num_samples_to_process_full,
      force_reprocess=True # Set to False if you want to load existing cache
  )

  print(f"\nSuccessfully created/loaded a fully enriched dataset with {len(full_enriched_dataset)} samples.")
  print("First sample from fully enriched dataset (should contain documentation and AST string):")
  print(full_enriched_dataset[0])

  # If the sample_record's hexsha was processed, its documentation should reflect the update
  if sample_record:
      print(f"\nChecking if locally updated doc for {original_hexsha[:10]}... is in the materialized dataset:")
      found_in_materialized = False
      for record in full_enriched_dataset:
          if record.get('hexsha') == original_hexsha:
              print(f"Materialized record {original_hexsha[:10]}... has documentation: {record['documentation']}")
              found_in_materialized = True
              break
      if not found_in_materialized:
          print(f"Record {original_hexsha[:10]}... was not found in the {num_samples_to_process_full} samples processed.")

In [26]:
import os
from datasets import Dataset

def create_and_cache_filtered_subset(filtered_dataset_instance: FilteredDataset, num_samples_to_cache: int, cache_dir: str = "./cached_data") -> Dataset:
    """
    Collects a specified number of filtered and processable samples from the streaming dataset,
    materializes them into a non-streaming Dataset, and saves them to disk.

    Args:
        filtered_dataset_instance (FilteredDataset): An instance of the FilteredDataset singleton.
        num_samples_to_cache (int): The number of valid samples to collect and cache.
        cache_dir (str): Directory to save the cached dataset.

    Returns:
        datasets.Dataset: The materialized and cached dataset.
    """
    os.makedirs(cache_dir, exist_ok=True)
    cached_file_path = os.path.join(cache_dir, f"filtered_subset_{num_samples_to_cache}")

    if os.path.exists(cached_file_path):
        print(f"Loading cached dataset from {cached_file_path}...")
        return Dataset.load_from_disk(cached_file_path)

    print(f"Generating and caching {num_samples_to_cache} samples...")
    raw_examples = []
    count = 0
    for example in filtered_dataset_instance:
        if example['is_processable_code']:
            # Optionally, perform heavy processing here if you want to cache the processed results
            # For this example, we're just caching the raw processable examples.

            # To avoid excessive memory usage, we'll store only relevant keys
            # You can customize which keys to keep based on your needs
            clean_example = {
                'hexsha': example.get('hexsha'),
                'content': example.get('content'),
                'lang': example.get('lang'),
                'language': example.get('language')
            }
            raw_examples.append(clean_example)
            count += 1
            if count >= num_samples_to_cache:
                break

    if not raw_examples:
        raise ValueError("No processable samples found to create a cached dataset.")

    print(f"Collected {len(raw_examples)} samples. Converting to Dataset...")
    cached_dataset = Dataset.from_list(raw_examples)
    cached_dataset.save_to_disk(cached_file_path)
    print(f"Cached dataset saved to {cached_file_path}")
    return cached_dataset

# Example usage:
# Ensure FilteredDataset is initialized
filtered_dataset = FilteredDataset()

# Create and cache a subset of 100 processable samples
num_samples = 100
cached_filtered_subset = create_and_cache_filtered_subset(filtered_dataset, num_samples)

print(f"\nSuccessfully created/loaded a cached dataset with {len(cached_filtered_subset)} samples.")
print("First 2 samples from cached dataset:")
for i in range(min(2, len(cached_filtered_subset))):
    print(cached_filtered_subset[i])

# Now, you can iterate over 'cached_filtered_subset' multiple times very efficiently.
# You can also apply further transformations or filtering to this non-streaming dataset.


Initializing FilteredDataset singleton...
Loading Python dataset...


Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

Loading C++ dataset...


Resolving data files:   0%|          | 0/110 [00:00<?, ?it/s]

FilteredDataset initialized successfully.
Generating and caching 100 samples...
Starting iteration over combined dataset (alternating Python and C++)...
Collected 100 samples. Converting to Dataset...


Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Cached dataset saved to ./cached_data/filtered_subset_100

Successfully created/loaded a cached dataset with 100 samples.
First 2 samples from cached dataset:
{'hexsha': 'd99a1e98eccb58cbc0c0cef6e9e6702f33461b0e', 'content': 'from rest_framework_gis import serializers\nfrom rest_framework import serializers as s\n\nfrom .models import (\n    Artificialisee2015to2018,\n    Artificielle2018,\n    CommunesSybarval,\n    CouvertureSol,\n    EnveloppeUrbaine2018,\n    Ocsge,\n    Renaturee2018to2015,\n    Sybarval,\n    Voirie2018,\n    ZonesBaties2018,\n    UsageSol,\n)\n\n\ndef get_label(code="", label=""):\n    if code is None:\n        code = "-"\n    if label is None:\n        label = "inconnu"\n    return f"{code} {label[:30]}"\n\n\nclass Artificialisee2015to2018Serializer(serializers.GeoFeatureModelSerializer):\n    usage_2015 = s.SerializerMethodField()\n    usage_2018 = s.SerializerMethodField()\n    couverture_2015 = s.SerializerMethodField()\n    couverture_2018 = s.SerializerMet

# Baseline
use the codegen-350m-multi model, and for each record in the dataset:
1. generte the documentation
2. give the model the documentation and ask it generate the code from the programming language of the record
3. create AST of the ground truth in the record and the generated program.
4. Compare the AST to generate the similarity score
5. Compare the generate code using CoderBERTScore

### Baseline Data Generation and Scoring

The `BaselineData` class orchestrates the generation of documentation, code, and subsequent scoring using AST similarity and GraphCodeBERT semantic similarity. It iterates through records, attempts multiple code generations from documentation, and stores the best-performing results for each record.

In [37]:
import torch
import random
from typing import Optional # Added import

# Ensure these singletons are imported so BaselineData can initialize them.
# from .code_documentation_generator import CodeDocumentationGenerator
# from .ast_processor import AST
# from .graphcodebert_scorer import GraphCodeBERTScorer
# from .filtered_dataset import FilteredDataset # Assuming FilteredDataset is already defined

class BaselineData(metaclass=SingletonMeta):
    _initialized = False # Class-level flag for singleton initialization

    def __init__(self):
        """
        Constructor for the BaselineData singleton class.
        Initializes all necessary components for baseline evaluation.
        """
        if BaselineData._initialized:
            print("BaselineData already initialized. Returning existing instance.")
            return
        BaselineData._initialized = True

        print("Initializing BaselineData singleton...")

        # Initialize dependencies internally as per user request to decouple FilteredDataset
        self._documentation_generator = CodeDocumentationGenerator()
        self._ast_processor = AST()
        self._graphcodebert_scorer = GraphCodeBERTScorer()
        # Initialize FilteredDataset without passing dependencies
        self._filtered_dataset = FilteredDataset()

        # Load codegen model for code generation from documentation
        self._load_codegen_model()

        print("BaselineData initialized successfully.")

    def _load_codegen_model(self):
        """
        Loads the Salesforce/codegen-350M-multi model for code generation.
        """
        from transformers import AutoTokenizer, AutoModelForCausalLM
        model_name_codegen = "Salesforce/codegen-350M-multi"
        print(f"Loading code generation model: {model_name_codegen}...")
        self._tokenizer_codegen = AutoTokenizer.from_pretrained(model_name_codegen)
        # Ensure pad_token_id is explicitly set, common for causal models where EOS is used for padding
        if self._tokenizer_codegen.pad_token_id is None:
            self._tokenizer_codegen.pad_token_id = self._tokenizer_codegen.eos_token_id
        self._model_codegen = AutoModelForCausalLM.from_pretrained(model_name_codegen, dtype=torch.float16, device_map="auto")
        print("Code generation model loaded.")

    def generate_code_from_documentation(self, documentation: str, language: str) -> str:
        """
        Generates code from given documentation using the codegen model.

        Args:
            documentation (str): The documentation to generate code from.
            language (str): The target programming language (e.g., 'python', 'cpp').

        Returns:
            str: The generated code.
        """
        # TODO: Refine prompt engineering for codegen-350M-multi
        prompt = f"""Generate {language} code based on the following documentation:
{documentation}
{language} code:"""

        # Encode the prompt and get both input_ids and attention_mask
        inputs = self._tokenizer_codegen(prompt, return_tensors="pt").to(self._model_codegen.device)
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]

        # Use appropriate generation parameters (e.g., max_new_tokens, do_sample, top_k, num_beams)
        # Adjust these for better quality/diversity of generated code
        output_ids = self._model_codegen.generate(
            input_ids,
            attention_mask=attention_mask, # Pass the attention mask explicitly
            max_new_tokens=256, # Limit generated code length
            do_sample=True, # Enable sampling for more diverse outputs
            top_k=50, # Consider top 50 tokens
            num_return_sequences=1,
            pad_token_id=self._tokenizer_codegen.eos_token_id # Important for handling padding
        )

        generated_text = self._tokenizer_codegen.decode(output_ids[0], skip_special_tokens=True)

        # Heuristic to extract only the code part
        # This might need refinement based on actual model output patterns
        code_prefix = f"{language} code:"
        if code_prefix in generated_text:
            generated_code = generated_text.split(code_prefix, 1)[1].strip()
        else:
            generated_code = generated_text.strip()

        # Further post-processing to remove any remaining prompt or unwanted text
        # e.g., if the model repeats the prompt or adds conversational text
        if documentation in generated_code:
            generated_code = generated_code.replace(documentation, "").strip()

        return generated_code


    def compute_baseline(self, num_records: Optional[int] = 100, num_tries: int = 3) -> list:
        """
        Computes a baseline score for a subset of the dataset.
        For each record, it generates documentation, then generates code multiple times
        from that documentation, and scores the best generated code.

        Args:
            num_records (Optional[int]): The number of records to process from the dataset.
                                        If None, processes all records.
            num_tries (int): The number of times to attempt code generation and scoring
                             for each record to find the best documentation/code pair.

        Returns:
            list: A list of dictionaries, each containing the hexsha, best AST score,
                  best GraphCodeBERT score, and average score for a processed record.
        """
        print(f"\nStarting baseline computation for {num_records if num_records is not None else 'all'} records (each with {num_tries} tries)...")
        results = []
        processed_count = 0

        # Reset the iterator to ensure we start from the beginning of the stream
        self._filtered_dataset.reset_iterator()

        for record in self._filtered_dataset:
            if num_records is not None and processed_count >= num_records: # Modified condition
                break

            if not record['is_processable_code']:
                continue # Skip unprocessable records

            original_hexsha = record['hexsha']
            original_code = record['content']
            language = record['language']

            best_ast_score = -1.0
            best_graphcodebert_score = -1.0
            best_average_score = -1.0
            best_documentation = ""

            # Generate documentation (only once per record)
            generated_doc = self._documentation_generator.generate_documentation(original_code)

            for _ in range(num_tries):
                # Generate code from the generated documentation
                generated_code = self.generate_code_from_documentation(generated_doc, language)

                if not generated_code.strip(): # Skip if no code was generated
                    continue

                try:
                    # 1. AST Similarity
                    original_ast = self._ast_processor.generate_ast(original_code, language)
                    generated_ast = self._ast_processor.generate_ast(generated_code, language)
                    ast_similarity = self._ast_processor.compare_ast(original_ast, generated_ast)
                except Exception as e:
                    print(f"Skipping AST comparison for {original_hexsha} due to error: {e}")
                    ast_similarity = 0.0 # Assign a low score if AST generation fails

                # 2. GraphCodeBERT Semantic Similarity
                try:
                    semantic_similarity = self._graphcodebert_scorer.score(original_code, generated_code)
                except Exception as e:
                    print(f"Skipping GCB comparison for {original_hexsha} due to error: {e}")
                    semantic_similarity = 0.0 # Assign a low score if GCB scoring fails

                current_average_score = (ast_similarity + semantic_similarity) / 2.0

                if current_average_score > best_average_score:
                    best_average_score = current_average_score
                    best_ast_score = ast_similarity
                    best_graphcodebert_score = semantic_similarity
                    best_documentation = generated_doc # Keep the documentation that led to this score

            if best_average_score > -1.0: # If at least one successful generation occurred
                # Update the FilteredDataset's internal caches with the best results
                self._filtered_dataset.update_documentation(original_hexsha, best_documentation)
                self._filtered_dataset.update_scores(original_hexsha, best_ast_score, best_graphcodebert_score, best_average_score)

                results.append({
                    'hexsha': original_hexsha,
                    'best_ast_score': best_ast_score,
                    'best_graphcodebert_score': best_graphcodebert_score,
                    'best_average_score': best_average_score
                })
                processed_count += 1
                print(f"Processed {processed_count}/{num_records if num_records is not None else 'all'} records. Current record {original_hexsha[:10]}... Best Avg Score: {best_average_score:.4f}")

        return results

In [40]:
from typing import Optional # Added import

print("\n--- Starting Baseline Evaluation for the entire dataset ---")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Instantiate the BaselineData singleton
baseline_evaluator = BaselineData()

# Set num_records_for_full_eval to None to process all records, or specify a number.
num_records_for_full_eval: Optional[int] = None # Process all records
# If you want to process a specific number of records for testing, uncomment and set the value:
num_records_for_full_eval = 50

num_generation_tries_eval = 3 # Number of tries for code generation per record

full_baseline_results = baseline_evaluator.compute_baseline(
    num_records=num_records_for_full_eval, # Pass None here
    num_tries=num_generation_tries_eval
)

print("\n--- Baseline Evaluation Complete ---")

# Calculate and display accuracy metrics
overall_scores = []
python_scores = []
cpp_scores = []

# Assuming `FilteredDataset` holds the up-to-date information after `compute_baseline`.
# We'll use the filtered_dataset_instance managed by BaselineData to retrieve language info.
# Make sure to reset its iterator to start fresh, and then iterate through it
# to get the language for each hexsha that was processed and has scores in `full_baseline_results`.
filtered_dataset_instance_for_metrics = baseline_evaluator._filtered_dataset
filtered_dataset_instance_for_metrics.reset_iterator()

# Create a dictionary to quickly look up scores by hexsha
results_by_hexsha = {res['hexsha']: res for res in full_baseline_results}

# Iterate through the filtered_dataset and collect scores for processed records
processed_for_metrics_count = 0
for record in filtered_dataset_instance_for_metrics:
    # We only care about records that were actually processed and have results
    hexsha = record.get('hexsha')
    if hexsha in results_by_hexsha:
        score_data = results_by_hexsha[hexsha]
        avg_score = score_data['best_average_score']
        language = record.get('language')

        overall_scores.append(avg_score)

        if language == 'python':
            python_scores.append(avg_score)
        elif language == 'cpp':
            cpp_scores.append(avg_score)
        processed_for_metrics_count += 1
        # If a specific number of records was processed, we stop once we have enough scores for metrics
        if num_records_for_full_eval is not None and processed_for_metrics_count >= num_records_for_full_eval:
             break


# Calculate averages
overall_average_accuracy = sum(overall_scores) / len(overall_scores) if overall_scores else 0.0
python_average_accuracy = sum(python_scores) / len(python_scores) if python_scores else 0.0
cpp_average_accuracy = sum(cpp_scores) / len(cpp_scores) if cpp_scores else 0.0

print("\n--- Final Accuracy Baselines ---")
print(f"Overall Average Code Generation Accuracy: {overall_average_accuracy:.4f} (based on {len(overall_scores)} records)")
print(f"Python Average Code Generation Accuracy:    {python_average_accuracy:.4f} (based on {len(python_scores)} records)")
print(f"C++ Average Code Generation Accuracy:       {cpp_average_accuracy:.4f} (based on {len(cpp_scores)} records)")


--- Starting Baseline Evaluation for the entire dataset ---
Initializing BaselineData singleton...
Loading code generation model: Salesforce/codegen-350M-multi...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Code generation model loaded.
BaselineData initialized successfully.

Starting baseline computation for 50 records (each with 3 tries)...
Resetting dataset iterator...
Loading Python dataset...


Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

Loading C++ dataset...


Resolving data files:   0%|          | 0/110 [00:00<?, ?it/s]

Starting iteration over combined dataset (alternating Python and C++)...


OutOfMemoryError: CUDA out of memory. Tried to allocate 166.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 95.81 MiB is free. Including non-PyTorch memory, this process has 14.47 GiB memory in use. Of the allocated memory 14.07 GiB is allocated by PyTorch, and 275.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

# Create LORA of codegen-multi-350B for code generation where the input is code documentation and output is the code.

In [ ]:
# Load the Codegen-350M-multi model
# Lora Adapt
# For the C++ and python code, get one, get the documentation, give model the documentation and let it generate the code.
# The generated code must be same as the code for which the documetation was generated.

# The second set of train is where we give documentation of a python code and have the model generate C++ Code:
# 1. Take Python code.
# 2. generate documentation,
# 3. Generated documentation input to model to genreate C++ Code
# 4. C++ Code as input to model to generate Python Code
# 5. Python code comparison gives the loss.


In [ ]:
# Load codegen-350m-multi model from hugging face


In [ ]:
if 'flag_baseline_data_test' in globals() and flag_baseline_data_test:
  # Example Usage:
  # Initialize the BaselineData singleton
  baseline_evaluator = BaselineData()

  # Run the baseline computation for a small number of records (e.g., 5 records, 2 tries each)
  num_records_to_process = 5
  num_generation_tries = 2

  print(f"\nStarting baseline evaluation for {num_records_to_process} records with {num_generation_tries} tries each...")

  baseline_results = baseline_evaluator.compute_baseline(
      num_records=num_records_to_process,
      num_tries=num_generation_tries
  )

  print("\nBaseline Evaluation Results:")
  for res in baseline_results:
      print(f"  Hexsha: {res['hexsha'][:10]}..., AST Score: {res['best_ast_score']:.4f}, GCB Score: {res['best_graphcodebert_score']:.4f}, Average Score: {res['best_average_score']:.4f}")

  # Verify that the FilteredDataset has been updated
  print("\nVerifying updates in FilteredDataset:")
  # Access the FilteredDataset instance managed by BaselineData
  filtered_dataset_instance = baseline_evaluator._filtered_dataset
  filtered_dataset_instance.reset_iterator()

  checked_count = 0
  for record in filtered_dataset_instance:
      if checked_count >= num_records_to_process: # Check first 'num_records_to_process' that are actually processable
          break
      # Only print records that were actually processed and updated by the baseline evaluator
      if record.get('hexsha') in [res['hexsha'] for res in baseline_results]:
          print(f"  Record {record['hexsha'][:10]}...: Doc updated? {bool(record['documentation'])}, AST Score: {record['ast_score']:.4f}, GCB Score: {record['graphcodebert_score']:.4f}, Avg Score: {record['average_score']:.4f}")
          checked_count += 1

### LORA Adaptation for `codegen-350M-multi`

Set up Low-Rank Adaptation (LORA) for the `codegen-350M-multi` model. This allows us to fine-tune the model efficiently without modifying all its parameters. We'll specify the LORA configuration, including the rank (`r`), alpha (`lora_alpha`), dropout (`lora_dropout`), and target modules (the layers to apply LORA to).


### Train the LORA Model

Set up the `Trainer` from the `transformers` library to fine-tune our LORA-adapted `codegen` model.
Define `TrainingArguments` to control the training process, such as the number of epochs, learning rate, and logging strategy.
Use a `DataCollatorForLanguageModeling` to handle batching and masking for language modeling tasks.

To save the best model based on validation accuracy , you would typically include a validation set and a custom `TrainerCallback` or configure `save_strategy='epoch'` and `load_best_model_at_end=True` with a specified `metric_for_best_model` in `TrainingArguments`.
